# Near-zero stiffness eigenvalues versus strain — SIM 5150 family

SIM 5150–5159 repeat the ST1000 neo-Hookean pressure sweep of SIM 5140–5149. The refined eigen replay samples 593 points: every 0.0025 through `t=0.84`, then every 0.000625 through `t=1`. At each point it saves four eigenpairs nearest zero in `spectrum.npz`; `mode0.npz` holds the selected eigenvalue history. The spatial mesh is unchanged.

The imposed 5 mm displacement over the 20 mm specimen gives nominal compressive strain `0.25 * t` (25% at `t=1`). This notebook plots the selected eigenvalue for all pressures, its first positive-to-nonpositive crossing, and the four overlap-matched branches for each run. Crossings before the eleventh interval are excluded, as in the 5140 notebook. Branch matching uses adjacent eigenvector overlap; a crossing with low overlap needs further inspection. These eigenvalues are nearest zero, not guaranteed to be the four algebraically smallest.

The 5150 eigen-only replay did not create `curves.npz`. For `f`, this notebook uses the matching SIM 5140–5149 lite curves, whose simulation parameters are identical apart from the input file path. It interpolates their efficiencies to each refined 5150 crossing time. These are reused physical-run results, not independent 5150 measurements.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

RESULTS_DIR = Path('../I001_Results')
if not RESULTS_DIR.exists():
    RESULTS_DIR = Path('I001_Results')

RESULTS_DIRS = [RESULTS_DIR]
fwd_file = RESULTS_DIR / 'AAA_fwd'
if fwd_file.is_file():
    forwarded_dir = Path(fwd_file.read_text().strip()).expanduser()
    if forwarded_dir.is_dir() and forwarded_dir.resolve() != RESULTS_DIR.resolve():
        RESULTS_DIRS.append(forwarded_dir)

def resolve_lite(sim_id, filename):
    relative_path = Path('LITE') / f'SIM_{sim_id}' / filename
    return next((directory / relative_path for directory in RESULTS_DIRS
                 if (directory / relative_path).is_file()), None)

SIM_IDS = range(5150, 5160)
PRESSURES = (0.0,) + tuple(0.04 * np.sqrt(2.0) ** i for i in range(9))
EXPECTED_POINTS = 593
MIN_CROSSING_LEFT_INDEX = 10
MAX_STRAIN = 5.0 / 20.0

print('Lite-result search order:')
for directory in RESULTS_DIRS:
    print(f'  {(directory / "LITE").resolve()}')
print(f'Nominal maximum compressive strain: {MAX_STRAIN:.1%}')

In [ ]:
def first_crossing(time, eigenvalue):
    indices = np.flatnonzero((eigenvalue[:-1] > 0.0) & (eigenvalue[1:] <= 0.0))
    indices = indices[indices >= MIN_CROSSING_LEFT_INDEX]
    if not len(indices):
        return np.nan
    i = int(indices[0])
    return time[i] - eigenvalue[i] * (time[i + 1] - time[i]) / (eigenvalue[i + 1] - eigenvalue[i])

def tracked_branches(values, overlaps):
    """Match each initial mode to subsequent modes using adjacent overlaps."""
    n_times, n_modes = values.shape
    indices = np.empty((n_times, n_modes), dtype=np.int16)
    confidence = np.ones((n_times, n_modes), dtype=np.float32)
    indices[0] = np.arange(n_modes)
    for i, overlap in enumerate(overlaps, start=1):
        rows, columns = linear_sum_assignment(-overlap)
        assignment = np.empty(n_modes, dtype=np.int16)
        assignment[rows] = columns
        indices[i] = assignment[indices[i - 1]]
        confidence[i] = overlap[indices[i - 1], indices[i]]
    return values[np.arange(n_times)[:, None], indices], confidence

curves = {}
missing = []
partial = []
invalid = []

for sim_id in SIM_IDS:
    mode_path = resolve_lite(sim_id, 'mode0.npz')
    spectrum_path = resolve_lite(sim_id, 'spectrum.npz')
    if mode_path is None or spectrum_path is None:
        missing.append((sim_id, [name for name, path in [('mode0.npz', mode_path),
                         ('spectrum.npz', spectrum_path)] if path is None]))
        continue
    try:
        # Access only named NPZ members; the large eigenvector array is not read.
        with np.load(mode_path, allow_pickle=False) as mode:
            time = np.asarray(mode['t'], dtype=float).copy()
            selected = np.asarray(mode['eigenvalue'], dtype=float).copy()
        with np.load(spectrum_path, allow_pickle=False) as spectrum:
            spectrum_time = np.asarray(spectrum['t'], dtype=float).copy()
            values = np.asarray(spectrum['eigenvalues'], dtype=float).copy()
            overlaps = np.asarray(spectrum['mode_overlap'], dtype=float).copy()
        n = len(time)
        if (time.ndim != 1 or selected.shape != (n,) or n < 2
                or spectrum_time.shape != (n,) or values.shape != (n, 4)
                or overlaps.shape != (n - 1, 4, 4)):
            raise ValueError('incompatible time, eigenvalue, or overlap shapes')
        if (not np.all(np.isfinite(time)) or not np.all(np.isfinite(selected))
                or not np.all(np.isfinite(values)) or not np.all(np.isfinite(overlaps))
                or not np.all(np.diff(time) > 0) or not np.allclose(time, spectrum_time)):
            raise ValueError('non-finite, unordered, or mismatched data')
        branches, confidence = tracked_branches(values, overlaps)
    except (KeyError, OSError, TypeError, ValueError) as error:
        invalid.append((sim_id, str(error)))
        continue

    complete = n == EXPECTED_POINTS and np.isclose(time[-1], 1.0)
    if not complete:
        partial.append((sim_id, n, time[-1]))
    curves[sim_id] = {
        'time': time, 'selected': selected, 'branches': branches,
        'confidence': confidence, 'pressure': PRESSURES[sim_id - 5150],
        'crossing_t': first_crossing(time, selected), 'complete': complete,
    }

print(f'Loaded refined eigenvalue histories: {len(curves)}/10')
for sim_id, data in sorted(curves.items()):
    print(f"  SIM {sim_id}: {len(data['time'])} points, p={data['pressure']:.6g} MPa")
if missing:
    print(f'Missing result files: {missing}')
if partial:
    print(f'Partial histories (SIM, points, final t): {partial}')
if invalid:
    print(f'Invalid histories: {invalid}')

In [ ]:
figures = {}

fig, ax = plt.subplots(figsize=(11, 7), constrained_layout=True)
for sim_id, data in sorted(curves.items()):
    label = f"SIM {sim_id}, p = {data['pressure']:.5g} MPa"
    if not data['complete']:
        label += f" (partial: {len(data['time'])} points)"
    line, = ax.plot(100 * MAX_STRAIN * data['time'], data['selected'],
                    '--' if not data['complete'] else '-', linewidth=1.4, label=label)
    if np.isfinite(data['crossing_t']):
        ax.scatter(100 * MAX_STRAIN * data['crossing_t'], 0,
                   color=line.get_color(), edgecolor='black', s=38, zorder=5)
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set(xlabel='Nominal compressive strain (%)', ylabel='Selected near-zero stiffness eigenvalue',
       title='SIM 5150 family: selected eigenvalue versus strain')
ax.grid(alpha=0.3)
if curves:
    ax.legend(fontsize=8)
figures['eigenvalue_strain'] = fig
plt.show()

crossing_rows = [(sim_id, data['pressure'], data['crossing_t'])
                 for sim_id, data in sorted(curves.items())
                 if np.isfinite(data['crossing_t'])]
print('First selected-mode positive-to-nonpositive crossings:')
print('  SIM     pressure (MPa)    time t    nominal strain (%)')
for sim_id, pressure, crossing_t in crossing_rows:
    print(f'  {sim_id:4d}    {pressure:12.6g}    {crossing_t:7.5f}    {100 * MAX_STRAIN * crossing_t:10.5f}')
if crossing_rows:
    fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
    ax.plot([row[1] for row in crossing_rows],
            [100 * MAX_STRAIN * row[2] for row in crossing_rows], 'o-')
    for sim_id, pressure, crossing_t in crossing_rows:
        ax.annotate(f'SIM {sim_id}', (pressure, 100 * MAX_STRAIN * crossing_t),
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax.set(xlabel='Internal pressure (MPa)',
           ylabel='Nominal strain at first crossing (%)',
           title='SIM 5150s: first selected-mode zero crossing')
    ax.grid(alpha=0.3)
    figures['crossing_pressure'] = fig
    plt.show()

fig, axes = plt.subplots(5, 2, figsize=(15, 18), sharex=True, constrained_layout=True)
branch_crossings = []
for sim_id, ax in zip(SIM_IDS, axes.flat):
    ax.set_title(f'SIM {sim_id}')
    ax.axhline(0, color='black', linewidth=0.7)
    if sim_id not in curves:
        ax.text(0.5, 0.5, 'No spectrum', ha='center', va='center', transform=ax.transAxes)
        continue
    data = curves[sim_id]
    time = data['time']
    for branch in range(4):
        eigenvalue = data['branches'][:, branch]
        ax.plot(100 * MAX_STRAIN * time, eigenvalue, linewidth=0.9,
                label=f'Initial mode {branch + 1}')
        indices = np.flatnonzero((eigenvalue[:-1] > 0) & (eigenvalue[1:] <= 0))
        indices = indices[indices >= MIN_CROSSING_LEFT_INDEX]
        for i in indices:
            fraction = eigenvalue[i] / (eigenvalue[i] - eigenvalue[i + 1])
            crossing_t = time[i] + fraction * (time[i + 1] - time[i])
            ax.plot(100 * MAX_STRAIN * crossing_t, 0, 'o', markersize=3)
            branch_crossings.append((sim_id, branch + 1, crossing_t,
                                     100 * MAX_STRAIN * crossing_t,
                                     float(data['confidence'][i + 1, branch])))
    ax.set_xlim(0, 25)
    ax.set_ylabel('Eigenvalue near zero')
    ax.grid(alpha=0.2)
for ax in axes[-1]:
    ax.set_xlabel('Nominal compressive strain (%)')
axes[0, 0].legend(fontsize=7)
figures['four_modes'] = fig
plt.show()
print(f'Observed branch crossings: {len(branch_crossings)}')
print('Each branch crossing records the adjacent-mode overlap in branch_crossings.')

In [ ]:
# The 5140 physical runs use the same input parameters as the 5150 family.
# Evaluate their efficiencies at the refined 5150 selected-mode crossing times.
f_rows = []
missing_efficiency = []
for sim_id, data in sorted(curves.items()):
    crossing_t = data['crossing_t']
    if not np.isfinite(crossing_t):
        continue
    source_sim = sim_id - 10
    curves_path = resolve_lite(source_sim, 'curves.npz')
    if curves_path is None:
        missing_efficiency.append((sim_id, 'missing matching 5140 curves'))
        continue
    try:
        with np.load(curves_path, allow_pickle=False) as saved:
            efficiency_time = np.asarray(saved['t'], dtype=float).copy()
            ef_t = np.asarray(saved['global_ef_t'], dtype=float).copy()
            ef_c = np.asarray(saved['global_ef_c'], dtype=float).copy()
        if (efficiency_time.ndim != 1 or len(efficiency_time) < 2
                or ef_t.shape != efficiency_time.shape or ef_c.shape != efficiency_time.shape
                or not np.all(np.isfinite(efficiency_time))
                or not np.all(np.isfinite(ef_t)) or not np.all(np.isfinite(ef_c))
                or not np.all(np.diff(efficiency_time) > 0)
                or not efficiency_time[0] <= crossing_t <= efficiency_time[-1]):
            raise ValueError('invalid or incomplete efficiency history')
        tension = float(np.interp(crossing_t, efficiency_time, ef_t))
        compression = float(np.interp(crossing_t, efficiency_time, ef_c))
        if compression == 0:
            raise ValueError('compression efficiency is zero')
        f_rows.append((sim_id, source_sim, data['pressure'], crossing_t,
                       tension / compression, tension, compression))
    except (KeyError, OSError, TypeError, ValueError) as error:
        missing_efficiency.append((sim_id, str(error)))

print('f at first refined selected-mode crossing (from matching 5140 physical run):')
print('  SIM   source    pressure (MPa)    strain (%)           f')
for sim_id, source_sim, pressure, crossing_t, f_value, tension, compression in f_rows:
    print(f'  {sim_id:4d}   {source_sim:4d}      {pressure:10.6g}      '
          f'{100 * MAX_STRAIN * crossing_t:9.5f}     {f_value:.8g}')
if missing_efficiency:
    print(f'Efficiency unavailable: {missing_efficiency}')

if f_rows:
    for key, x_index, xlabel, title_tail in [
        ('f_pressure', 2, 'Internal pressure (MPa)', 'pressure'),
        ('f_strain', 3, 'Nominal strain at first crossing (%)', 'crossing strain'),
    ]:
        fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
        x = [100 * MAX_STRAIN * row[x_index] if key == 'f_strain' else row[x_index]
             for row in f_rows]
        y = [row[4] for row in f_rows]
        ax.plot(x, y, 'o-')
        for position, row in zip(x, f_rows):
            ax.annotate(f'SIM {row[0]}', (position, row[4]),
                        xytext=(5, 5), textcoords='offset points', fontsize=8)
        ax.set(xlabel=xlabel, ylabel='f = global tension efficiency / compression efficiency',
               title=f'SIM 5150s: f at refined eigenvalue zero versus {title_tail}')
        ax.grid(alpha=0.3)
        figures[key] = fig
        plt.show()

In [ ]:
# Save figures next to this notebook when run from its directory.
output_names = {
    'eigenvalue_strain': 'SIM_5150s_selected_eigenvalue_vs_strain.png',
    'crossing_pressure': 'SIM_5150s_first_eigenvalue_zero_crossing_vs_pressure.png',
    'four_modes': 'SIM_5150s_four_eigenvalues_vs_strain.png',
    'f_pressure': 'SIM_5150s_f_at_eigenvalue_zero_vs_pressure.png',
    'f_strain': 'SIM_5150s_f_at_eigenvalue_zero_vs_strain.png',
}
for key, figure in figures.items():
    output_path = Path(output_names[key])
    figure.savefig(output_path, dpi=200, bbox_inches='tight')
    print(f'Saved: {output_path.resolve()}')